# Generative Adversarial Networks

CSCI 6379 &middot; Topic 23. Two networks play a game: a **generator** turns noise into images, and a
**discriminator** tries to tell those images from real ones. Neither is ever shown how to draw a digit.

This notebook is the code from the lecture, in the order it appears there. Run it top to bottom.

**Set the runtime to a GPU**: Runtime &rarr; Change runtime type &rarr; T4 GPU. It will run on CPU, but slowly.


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
Z, BATCH, LR, IMG = 100, 128, 2e-4, 28 * 28
torch.manual_seed(0)
print('device:', device)


## The data: everything lives in [-1, 1]

`ToTensor()` puts the pixels in `[0, 1]`, and `Normalize((0.5,), (0.5,))` shifts that to `[-1, 1]`.

This is not cosmetic. The generator's last layer is `Tanh`, which can only ever emit `[-1, 1]`.
Moving the real images into the same range is what makes it possible for the generator to match
them at all.

`drop_last=True` throws away the short final batch, so every batch is exactly 128 and the label
tensors never have to be rebuilt.


In [ ]:
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

loader = DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=tf),
    batch_size=BATCH, shuffle=True, drop_last=True,
)
print('batches per epoch:', len(loader))


## The two networks

The generator grows a 100-number noise vector into 784 pixels. The discriminator squeezes 784
pixels down to a single number.

Note there is **no `Sigmoid`** at the end of `D`. It emits a raw **logit**, and the sigmoid lives
inside `BCEWithLogitsLoss`. Same rule as `CrossEntropyLoss`: the loss carries the activation.


In [ ]:
def hidden(i, o):
    return [nn.Linear(i, o), nn.LeakyReLU(0.2)]

G = nn.Sequential(*hidden(Z, 256), *hidden(256, 512), *hidden(512, 1024),
                  nn.Linear(1024, IMG), nn.Tanh()).to(device)

D = nn.Sequential(*hidden(IMG, 512), *hidden(512, 256),
                  nn.Linear(256, 1)).to(device)

loss = nn.BCEWithLogitsLoss()
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

print('generator parameters    :', sum(t.numel() for t in G.parameters()))
print('discriminator parameters:', sum(t.numel() for t in D.parameters()))


## The training step, and the one line that matters

The same `fake` tensor is used twice: **detached** for the discriminator, **attached** for the
generator.

Without `.detach()`, the discriminator's loss would flow back into the generator and train it to
*help* `D` catch its own fakes. That single call is what keeps the two players apart.

Note also that the generator labels its own fakes **`ones`**, that is, "real". That looks like a
typo the first time you read it. It is the **non-saturating** form of the generator loss: the
textbook objective minimizes `log(1 - D(G(z)))`, whose gradient dies exactly when the generator is
still bad, while maximizing `log D(G(z))` has the same optimum and a far stronger gradient early on.

The lecture used `EPOCHS = 30`. That took 7 minutes 49 seconds on an RTX 4090. Start with 5 here
and raise it if you have time; a Colab GPU is slower than a 4090, so measure before you commit to 30.


In [ ]:
EPOCHS = 5

G_hist, D_hist = [], []
for epoch in range(EPOCHS):
    for real, _ in loader:
        real = real.view(BATCH, IMG).to(device)
        ones = torch.ones(BATCH, 1, device=device)
        zeros = torch.zeros(BATCH, 1, device=device)

        fake = G(torch.randn(BATCH, Z, device=device))

        # the discriminator wants real -> 1 and fake -> 0
        d_loss = 0.5 * (loss(D(real), ones) + loss(D(fake.detach()), zeros))
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

        # the generator wants the discriminator to call its fakes real
        g_loss = loss(D(fake), ones)
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        G_hist.append(g_loss.item()); D_hist.append(d_loss.item())
    print(f'epoch {epoch+1:>3}/{EPOCHS}  D {d_loss.item():.4f}  G {g_loss.item():.4f}')


## What the losses should look like

Neither loss goes to zero, and neither should. The discriminator settles near `log 2 = 0.69`,
which is what a coin flip costs. That is the sign of success: it can no longer tell the two apart.
A GAN whose discriminator loss collapses to zero has a generator that stopped learning.


In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(G_hist, lw=0.5, alpha=0.8, label='generator loss')
plt.plot(D_hist, lw=0.5, alpha=0.8, label='discriminator loss')
plt.xlabel('iteration'); plt.ylabel('BCE loss'); plt.legend(); plt.grid(alpha=0.2)
plt.show()

with torch.no_grad():
    samples = G(torch.randn(25, Z, device=device)).view(25, 1, 28, 28)
grid = utils.make_grid(samples, nrow=5, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(4, 4))
plt.imshow(grid.cpu().permute(1, 2, 0)); plt.axis('off'); plt.show()


# DCGAN: the same game, with convolutions

Only the two networks change. The data, the loss and the training step are identical.

`ConvTranspose2d` is convolution run backwards: it takes one number and paints a whole
kernel-sized patch, then adds the patches up. Three of them grow `100 x 1 x 1` into `1 x 28 x 28`,
each step following `(n - 1)s - 2p + k`:

`(1-1)*1 - 0 + 7 = 7`, then `(7-1)*2 - 2 + 4 = 14`, then `(14-1)*2 - 2 + 4 = 28`.


In [ ]:
def up(i, o, k, s, p):
    return [nn.ConvTranspose2d(i, o, k, s, p, bias=False), nn.BatchNorm2d(o), nn.ReLU(True)]

def down(i, o, k, s, p, norm=True):
    layers = [nn.Conv2d(i, o, k, s, p, bias=False)]
    if norm:
        layers.append(nn.BatchNorm2d(o))
    return layers + [nn.LeakyReLU(0.2, inplace=True)]

G2 = nn.Sequential(*up(Z, 128, 7, 1, 0),      # 100 x 1 x 1  -> 128 x  7 x  7
                   *up(128, 64, 4, 2, 1),     #              -> 64  x 14 x 14
                   nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),   # -> 1 x 28 x 28
                   nn.Tanh()).to(device)

D2 = nn.Sequential(*down(1, 64, 4, 2, 1, norm=False),
                   *down(64, 128, 4, 2, 1),
                   nn.Conv2d(128, 1, 7, 1, 0, bias=False),
                   nn.Flatten()).to(device)

opt_G2 = torch.optim.Adam(G2.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D2 = torch.optim.Adam(D2.parameters(), lr=LR, betas=(0.5, 0.999))

print('generator parameters    :', sum(t.numel() for t in G2.parameters()))
print('discriminator parameters:', sum(t.numel() for t in D2.parameters()))
print('one forward pass:', G2(torch.randn(2, Z, 1, 1, device=device)).shape)


## Train it the same way

This loop is the dense one with two changes: the images stay as `1 x 28 x 28` instead of being
flattened, and the noise is `Z x 1 x 1` instead of a flat vector. Everything about the game is the same.


In [ ]:
G2_hist, D2_hist = [], []
for epoch in range(EPOCHS):
    for real, _ in loader:
        real = real.to(device)
        ones = torch.ones(BATCH, 1, device=device)
        zeros = torch.zeros(BATCH, 1, device=device)

        fake = G2(torch.randn(BATCH, Z, 1, 1, device=device))

        d_loss = 0.5 * (loss(D2(real), ones) + loss(D2(fake.detach()), zeros))
        opt_D2.zero_grad(); d_loss.backward(); opt_D2.step()

        g_loss = loss(D2(fake), ones)
        opt_G2.zero_grad(); g_loss.backward(); opt_G2.step()

        G2_hist.append(g_loss.item()); D2_hist.append(d_loss.item())
    print(f'epoch {epoch+1:>3}/{EPOCHS}  D {d_loss.item():.4f}  G {g_loss.item():.4f}')

with torch.no_grad():
    samples2 = G2(torch.randn(25, Z, 1, 1, device=device))
grid2 = utils.make_grid(samples2, nrow=5, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(4, 4))
plt.imshow(grid2.cpu().permute(1, 2, 0)); plt.axis('off'); plt.show()


## What to notice

At 30 epochs on an RTX 4090 the lecture measured:

| | generator params | discriminator params | per epoch | 30 epochs |
|---|---|---|---|---|
| GAN | 1,486,352 | 533,505 | 15.6 s | 7 min 49 s |
| DCGAN | 759,680 | 138,624 | 15.8 s | 7 min 53 s |

DCGAN has less than half the parameters, produces visibly cleaner strokes, and takes the same
wall-clock time. **Parameter count is not compute**: a convolution reuses each weight at every
position, so it does far more arithmetic per parameter than a dense layer does.
